# 📊 MODEL EVALUATION & COMPARISON

This notebook evaluates forecasting performance by:
1. Loading test data and trained models
2. Implementing baseline models (Naive, Weighted MA)
3. Generating forecasts for all products and dates
4. Calculating comprehensive evaluation metrics
5. Comparing performance across product segments
6. Creating visualizations

---

## STEP 1: LOAD MODELS AND DATA

In [ ]:
# ===================================================================
# LOAD TEST DATA, MODELS, AND SEGMENTS
# ===================================================================

import pandas as pd
import numpy as np
import pickle
from sklearn.metrics import mean_absolute_error, mean_squared_error
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print("Loading models and data...")

# Load test data
test_df = pd.read_csv('../data/processed/test.csv')
test_df['date'] = pd.to_datetime(test_df['date'])

# Load product segments
product_segments = pd.read_csv('../data/processed/product_segments.csv')

# Load Random Forest model
with open('../data/models/random_forest_model.pkl', 'rb') as f:
    rf_model = pickle.load(f)

with open('../data/models/feature_columns.pkl', 'rb') as f:
    feature_cols = pickle.load(f)

print(f"Test set: {len(test_df):,} rows")
print(f"Date range: {test_df['date'].min().date()} to {test_df['date'].max().date()}")
print(f"Unique test dates: {test_df['date'].nunique()}")
print(f"Products: {test_df['id_produit'].nunique()}")
print(f"Product segments: {product_segments['segment'].nunique()}")

## STEP 2: IMPLEMENT BASELINE MODELS

In [ ]:
# ===================================================================
# BASELINE FORECASTING METHODS
# ===================================================================

print("\n" + "="*60)
print("STEP 2: Implementing baseline forecasters...")
print("="*60)

def naive_forecast(df, product_id, forecast_date, window=7):
    """
    Naive Baseline: Average of last N days
    """
    product_data = df[df['id_produit'] == product_id].copy()
    product_data = product_data.sort_values('date')
    
    recent_data = product_data[
        product_data['date'] < forecast_date
    ].tail(window)
    
    if len(recent_data) == 0:
        return 0
    
    forecast = recent_data['quantite_demande'].mean()
    return max(0, round(forecast))


def weighted_moving_average(df, product_id, forecast_date, window=14):
    """
    Weighted Moving Average: Recent days weighted more
    """
    product_data = df[df['id_produit'] == product_id].copy()
    product_data = product_data.sort_values('date')
    
    recent_data = product_data[
        product_data['date'] < forecast_date
    ].tail(window)
    
    if len(recent_data) < 3:
        return naive_forecast(df, product_id, forecast_date, window=7)
    
    # Exponential weights
    weights = np.exp(np.linspace(-1, 0, len(recent_data)))
    weights = weights / weights.sum()
    
    forecast = np.average(recent_data['quantite_demande'], weights=weights)
    
    # Trend adjustment
    if len(recent_data) >= 7:
        first_half = recent_data.iloc[:len(recent_data)//2]['quantite_demande'].mean()
        second_half = recent_data.iloc[len(recent_data)//2:]['quantite_demande'].mean()
        trend = second_half - first_half
        forecast = forecast + (trend * 0.3)
    
    return max(0, round(forecast))


def predict_random_forest(model, df, product_id, forecast_date, feature_cols):
    """
    Random Forest prediction
    """
    product_data = df[df['id_produit'] == product_id].copy()
    product_data = product_data.sort_values('date')
    
    latest = product_data[product_data['date'] < forecast_date].tail(1)
    
    if len(latest) == 0:
        return 0
    
    # Update temporal features for forecast date
    latest_copy = latest.copy()
    latest_copy['day_of_week'] = forecast_date.weekday()
    latest_copy['month'] = forecast_date.month
    latest_copy['week_of_year'] = forecast_date.isocalendar()[1]
    latest_copy['is_weekend'] = 1 if forecast_date.weekday() >= 5 else 0
    latest_copy['is_month_start'] = 1 if forecast_date.day <= 7 else 0
    latest_copy['is_month_end'] = 1 if forecast_date.day >= 24 else 0
    
    # Update cyclical features if they exist
    if 'month_sin' in feature_cols:
        latest_copy['month_sin'] = np.sin(2 * np.pi * forecast_date.month / 12)
        latest_copy['month_cos'] = np.cos(2 * np.pi * forecast_date.month / 12)
    if 'dow_sin' in feature_cols:
        latest_copy['dow_sin'] = np.sin(2 * np.pi * forecast_date.weekday() / 7)
        latest_copy['dow_cos'] = np.cos(2 * np.pi * forecast_date.weekday() / 7)
    
    try:
        X_pred = latest_copy[feature_cols]
        prediction = model.predict(X_pred)[0]
        return max(0, round(prediction))
    except:
        return weighted_moving_average(df, product_id, forecast_date)

print("✓ Baseline models defined")

## STEP 3: GENERATE FORECASTS FOR ALL TEST DATES

In [ ]:
# ===================================================================
# GENERATE FORECASTS FOR TEST PERIOD
# ===================================================================

print("\n" + "="*60)
print("STEP 3: Generating forecasts for test period...")
print("="*60)

# Combine train + test data (needed for lag features)
full_df = pd.concat([
    pd.read_csv('../data/processed/train.csv'),
    test_df
], ignore_index=True)
full_df['date'] = pd.to_datetime(full_df['date'])

# Get unique test dates
test_dates = sorted(test_df['date'].unique())
print(f"Forecasting for {len(test_dates)} days...")

# Get all products
all_products = test_df['id_produit'].unique()
print(f"Products: {len(all_products)}")

# Storage for predictions
predictions = []

for i, forecast_date in enumerate(test_dates):
    print(f"\rForecasting for {forecast_date.date()} ({i+1}/{len(test_dates)})...", end='')
    
    for product_id in all_products:
        # Get product segment
        segment_info = product_segments[product_segments['id_produit'] == product_id]
        
        if len(segment_info) == 0:
            segment = 'E_LOW_FREQ'
        else:
            segment = segment_info.iloc[0]['segment']
        
        # Generate forecasts with different methods
        
        # NAIVE BASELINE (for comparison)
        naive_pred = naive_forecast(full_df, product_id, forecast_date, window=7)
        
        # PROPOSED MODEL (segment-based)
        if segment == 'A_VERY_HIGH_FREQ_HIGH_VOL':
            proposed_pred = predict_random_forest(
                rf_model, full_df, product_id, forecast_date, feature_cols
            )
            method = 'Random Forest'
        elif segment in ['B_VERY_HIGH_FREQ_LOW_VOL', 'C_HIGH_FREQ']:
            proposed_pred = weighted_moving_average(
                full_df, product_id, forecast_date, window=14
            )
            method = 'Weighted MA'
        else:
            proposed_pred = naive_forecast(
                full_df, product_id, forecast_date, window=7
            )
            method = 'Naive'
        
        # Get actual demand
        actual = test_df[
            (test_df['id_produit'] == product_id) & 
            (test_df['date'] == forecast_date)
        ]['quantite_demande'].values
        
        actual_demand = actual[0] if len(actual) > 0 else 0
        
        predictions.append({
            'date': forecast_date,
            'id_produit': product_id,
            'segment': segment,
            'actual': actual_demand,
            'naive_forecast': naive_pred,
            'proposed_forecast': proposed_pred,
            'method': method
        })

predictions_df = pd.DataFrame(predictions)

print("\n\nForecasting complete!")
print(f"Total predictions: {len(predictions_df):,}")

# Save predictions
predictions_df.to_csv('../data/evaluation/predictions.csv', index=False)
print("✓ Saved: ../data/evaluation/predictions.csv")

## STEP 4: CALCULATE EVALUATION METRICS

In [ ]:
# ===================================================================
# CALCULATE FORECAST ACCURACY METRICS
# ===================================================================

print("\n" + "="*60)
print("STEP 4: Calculating evaluation metrics...")
print("="*60)

def calculate_metrics(actual, predicted):
    """
    Calculate forecast accuracy metrics
    """
    # MAE (Mean Absolute Error)
    mae = mean_absolute_error(actual, predicted)
    
    # RMSE (Root Mean Squared Error)
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    
    # MAPE (Mean Absolute Percentage Error)
    # Avoid division by zero
    mape = np.mean(
        np.abs((actual - predicted) / (actual + 1))
    ) * 100
    
    # Bias (Over/Under forecasting)
    bias = np.mean(predicted - actual)
    
    # Service Level (% demand satisfied)
    satisfied = np.minimum(predicted, actual)
    service_level = satisfied.sum() / (actual.sum() + 1) * 100
    
    return {
        'MAE': mae,
        'RMSE': rmse,
        'MAPE': mape,
        'Bias': bias,
        'Service_Level': service_level,
        'Total_Actual': actual.sum(),
        'Total_Forecast': predicted.sum()
    }

# Overall metrics
naive_metrics = calculate_metrics(
    predictions_df['actual'].values,
    predictions_df['naive_forecast'].values
)

proposed_metrics = calculate_metrics(
    predictions_df['actual'].values,
    predictions_df['proposed_forecast'].values
)

print("\n" + "="*80)
print("OVERALL FORECAST ACCURACY")
print("="*80)

print("\n📊 NAIVE BASELINE (7-day average):")
for metric, value in naive_metrics.items():
    print(f"  {metric:20s}: {value:,.2f}")

print("\n🚀 PROPOSED MODEL (Hybrid approach):")
for metric, value in proposed_metrics.items():
    print(f"  {metric:20s}: {value:,.2f}")

print("\n📈 IMPROVEMENT:")
print(f"  MAE improvement:     {(1 - proposed_metrics['MAE']/naive_metrics['MAE'])*100:,.1f}%")
print(f"  RMSE improvement:    {(1 - proposed_metrics['RMSE']/naive_metrics['RMSE'])*100:,.1f}%")
print(f"  MAPE improvement:    {naive_metrics['MAPE'] - proposed_metrics['MAPE']:,.2f} pp")
print(f"  Service Level gain:  {proposed_metrics['Service_Level'] - naive_metrics['Service_Level']:,.2f} pp")

## STEP 5: METRICS BY SEGMENT

In [ ]:
# ===================================================================
# SEGMENT-WISE PERFORMANCE ANALYSIS
# ===================================================================

print("\n" + "="*60)
print("STEP 5: Metrics by product segment...")
print("="*60)

segment_results = []

for segment in predictions_df['segment'].unique():
    segment_data = predictions_df[predictions_df['segment'] == segment]
    
    naive_seg = calculate_metrics(
        segment_data['actual'].values,
        segment_data['naive_forecast'].values
    )
    
    proposed_seg = calculate_metrics(
        segment_data['actual'].values,
        segment_data['proposed_forecast'].values
    )
    
    segment_results.append({
        'Segment': segment,
        'Products': segment_data['id_produit'].nunique(),
        'Naive_MAE': naive_seg['MAE'],
        'Proposed_MAE': proposed_seg['MAE'],
        'Improvement_%': (1 - proposed_seg['MAE']/naive_seg['MAE'])*100,
        'Naive_MAPE': naive_seg['MAPE'],
        'Proposed_MAPE': proposed_seg['MAPE'],
        'Service_Level': proposed_seg['Service_Level']
    })

segment_results_df = pd.DataFrame(segment_results)
print("\nSegment-wise Performance:")
print(segment_results_df.to_string(index=False))

segment_results_df.to_csv('../data/evaluation/segment_performance.csv', index=False)
print("\n✓ Saved: ../data/evaluation/segment_performance.csv")

## STEP 6: VISUALIZATIONS

In [ ]:
# ===================================================================
# PERFORMANCE VISUALIZATIONS
# ===================================================================

print("\n" + "="*60)
print("STEP 6: Creating visualizations...")
print("="*60)

# 1. Overall comparison
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# MAE comparison
metrics_comparison = pd.DataFrame({
    'Metric': ['MAE', 'RMSE', 'MAPE', 'Service Level'],
    'Naive': [naive_metrics['MAE'], naive_metrics['RMSE'], 
              naive_metrics['MAPE'], naive_metrics['Service_Level']],
    'Proposed': [proposed_metrics['MAE'], proposed_metrics['RMSE'], 
                 proposed_metrics['MAPE'], proposed_metrics['Service_Level']]
})

x = np.arange(len(metrics_comparison))
width = 0.35
axes[0, 0].bar(x - width/2, metrics_comparison['Naive'], width, label='Naive Baseline', alpha=0.8)
axes[0, 0].bar(x + width/2, metrics_comparison['Proposed'], width, label='Proposed Model', alpha=0.8)
axes[0, 0].set_xticks(x)
axes[0, 0].set_xticklabels(metrics_comparison['Metric'], rotation=0)
axes[0, 0].set_title('Metric Comparison: Naive vs Proposed', fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Scatter: Actual vs Predicted (Proposed)
axes[0, 1].scatter(
    predictions_df['actual'], 
    predictions_df['proposed_forecast'],
    alpha=0.3, s=10
)
max_val = predictions_df['actual'].max()
axes[0, 1].plot([0, max_val], [0, max_val], 'r--', label='Perfect prediction', linewidth=2)
axes[0, 1].set_xlabel('Actual Demand')
axes[0, 1].set_ylabel('Predicted Demand')
axes[0, 1].set_title('Proposed Model: Actual vs Predicted', fontweight='bold')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Error distribution
errors = predictions_df['proposed_forecast'] - predictions_df['actual']
axes[1, 0].hist(errors, bins=50, edgecolor='black', alpha=0.7)
axes[1, 0].axvline(x=0, color='r', linestyle='--', label='Zero error', linewidth=2)
axes[1, 0].set_xlabel('Prediction Error')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].set_title('Error Distribution (Proposed Model)', fontweight='bold')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Segment performance
axes[1, 1].bar(range(len(segment_results_df)), segment_results_df['Improvement_%'], alpha=0.7)
axes[1, 1].set_xticks(range(len(segment_results_df)))
axes[1, 1].set_xticklabels(segment_results_df['Segment'], rotation=45, ha='right')
axes[1, 1].set_xlabel('Product Segment')
axes[1, 1].set_ylabel('Improvement over Naive (%)')
axes[1, 1].set_title('Improvement by Segment', fontweight='bold')
axes[1, 1].axhline(y=0, color='r', linestyle='--', alpha=0.5)
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../data/evaluation/overall_performance.png', dpi=300, bbox_inches='tight')
print("  ✓ Saved: overall_performance.png")
plt.show()

In [ ]:
# 2. Time series plot for top products
print("\nCreating time series plots for top products...")

top_products = predictions_df.groupby('id_produit')['actual'].sum().nlargest(5).index

fig, axes = plt.subplots(5, 1, figsize=(14, 12))

for i, product in enumerate(top_products):
    product_pred = predictions_df[predictions_df['id_produit'] == product].sort_values('date')
    
    axes[i].plot(product_pred['date'], product_pred['actual'], 
                 label='Actual', marker='o', linewidth=2, markersize=6)
    axes[i].plot(product_pred['date'], product_pred['naive_forecast'], 
                 label='Naive', marker='s', linestyle='--', alpha=0.7, markersize=4)
    axes[i].plot(product_pred['date'], product_pred['proposed_forecast'], 
                 label='Proposed', marker='^', linestyle='--', alpha=0.7, markersize=4)
    
    axes[i].set_title(f'Product {product}', fontweight='bold')
    axes[i].set_ylabel('Demand')
    axes[i].legend(loc='upper right')
    axes[i].grid(True, alpha=0.3)
    
    if i < 4:
        axes[i].set_xticklabels([])

plt.tight_layout()
plt.savefig('../data/evaluation/top_products_forecast.png', dpi=300, bbox_inches='tight')
print("  ✓ Saved: top_products_forecast.png")
plt.show()

print("\n" + "="*60)
print("✅ EVALUATION COMPLETE!")
print("="*60)

In [ ]:
# ===================================================================
# VISUALIZATION 6: NAIVE VS PROPOSED COMPARISON
# ===================================================================

print("\nCreating Naive vs Proposed comparison visualizations...")

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 1. Absolute errors comparison
naive_errors = np.abs(predictions_df['naive_forecast'] - predictions_df['actual'])
proposed_errors = np.abs(predictions_df['proposed_forecast'] - predictions_df['actual'])

axes[0, 0].scatter(naive_errors, proposed_errors, alpha=0.3, s=10)
max_err = max(naive_errors.max(), proposed_errors.max())
axes[0, 0].plot([0, max_err], [0, max_err], 'r--', linewidth=2, label='Equal error')
axes[0, 0].set_xlabel('Naive Absolute Error')
axes[0, 0].set_ylabel('Proposed Absolute Error')
axes[0, 0].set_title('Error Comparison: Naive vs Proposed', fontsize=11, fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# 2. Error improvement distribution
improvement = naive_errors - proposed_errors
axes[0, 1].hist(improvement, bins=50, edgecolor='black', alpha=0.7)
axes[0, 1].axvline(x=0, color='r', linestyle='--', linewidth=2, label='No improvement')
axes[0, 1].set_xlabel('Error Reduction (Naive - Proposed)')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title('Error Improvement Distribution', fontsize=11, fontweight='bold')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# 3. Cumulative demand comparison
axes[1, 0].plot(range(len(predictions_df)), 
                predictions_df['actual'].cumsum().values, 
                label='Actual', linewidth=2)
axes[1, 0].plot(range(len(predictions_df)), 
                predictions_df['naive_forecast'].cumsum().values, 
                label='Naive', linewidth=1.5, linestyle='--')
axes[1, 0].plot(range(len(predictions_df)), 
                predictions_df['proposed_forecast'].cumsum().values, 
                label='Proposed', linewidth=1.5, linestyle='--')
axes[1, 0].set_xlabel('Prediction Index')
axes[1, 0].set_ylabel('Cumulative Demand')
axes[1, 0].set_title('Cumulative Demand Comparison', fontsize=11, fontweight='bold')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# 4. Win rate by error threshold
thresholds = [5, 10, 20, 50, 100]
naive_wins = []
proposed_wins = []
ties = []

for threshold in thresholds:
    naive_win = ((naive_errors <= threshold) & (proposed_errors > threshold)).sum()
    proposed_win = ((proposed_errors <= threshold) & (naive_errors > threshold)).sum()
    tie = ((naive_errors <= threshold) & (proposed_errors <= threshold)).sum()
    
    naive_wins.append(naive_win)
    proposed_wins.append(proposed_win)
    ties.append(tie)

x = np.arange(len(thresholds))
width = 0.25
axes[1, 1].bar(x - width, naive_wins, width, label='Naive Wins', alpha=0.8)
axes[1, 1].bar(x, proposed_wins, width, label='Proposed Wins', alpha=0.8)
axes[1, 1].bar(x + width, ties, width, label='Both Good', alpha=0.8)
axes[1, 1].set_xticks(x)
axes[1, 1].set_xticklabels([f'±{t}' for t in thresholds])
axes[1, 1].set_xlabel('Error Threshold')
axes[1, 1].set_ylabel('Count')
axes[1, 1].set_title('Win Rate by Error Threshold', fontsize=11, fontweight='bold')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("  ✓ Naive vs Proposed comparison visualizations complete")
print("\n✓ ALL EVALUATION VISUALIZATIONS COMPLETE!")

In [ ]:
# ===================================================================
# VISUALIZATION 5: PRODUCT-WISE PERFORMANCE DISTRIBUTION
# ===================================================================

print("\nCreating product-wise performance visualizations...")

# Calculate MAE per product
product_mae = []
for product_id in predictions_df['id_produit'].unique():
    prod_data = predictions_df[predictions_df['id_produit'] == product_id]
    mae = mean_absolute_error(prod_data['actual'], prod_data['proposed_forecast'])
    total_actual = prod_data['actual'].sum()
    product_mae.append({
        'id_produit': product_id,
        'mae': mae,
        'total_actual': total_actual
    })
product_mae_df = pd.DataFrame(product_mae)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. MAE distribution
axes[0].hist(product_mae_df['mae'], bins=50, edgecolor='black', alpha=0.7)
axes[0].set_title('Distribution of Product-Level MAE', fontsize=11, fontweight='bold')
axes[0].set_xlabel('MAE')
axes[0].set_ylabel('Number of Products')
axes[0].set_yscale('log')
axes[0].grid(True, alpha=0.3)

# 2. MAE vs Total Demand
axes[1].scatter(product_mae_df['total_actual'], product_mae_df['mae'], alpha=0.5, s=30)
axes[1].set_title('MAE vs Total Demand', fontsize=11, fontweight='bold')
axes[1].set_xlabel('Total Actual Demand')
axes[1].set_ylabel('MAE')
axes[1].set_xscale('log')
axes[1].set_yscale('log')
axes[1].grid(True, alpha=0.3)

# 3. Top 10 worst performers
worst_products = product_mae_df.nlargest(10, 'mae')
axes[2].barh(range(len(worst_products)), worst_products['mae'].values, alpha=0.7, color='coral')
axes[2].set_yticks(range(len(worst_products)))
axes[2].set_yticklabels([f'Prod {pid}' for pid in worst_products['id_produit']])
axes[2].set_title('Top 10 Products by MAE', fontsize=11, fontweight='bold')
axes[2].set_xlabel('MAE')
axes[2].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

print("  ✓ Product-wise performance visualizations complete")

In [ ]:
# ===================================================================
# VISUALIZATION 4: FORECAST ACCURACY OVER TIME
# ===================================================================

print("\nCreating time-based accuracy visualizations...")

# Calculate daily metrics
daily_metrics = []
for date in predictions_df['date'].unique():
    date_data = predictions_df[predictions_df['date'] == date]
    mae = mean_absolute_error(date_data['actual'], date_data['proposed_forecast'])
    rmse = np.sqrt(mean_squared_error(date_data['actual'], date_data['proposed_forecast']))
    daily_metrics.append({
        'date': date,
        'mae': mae,
        'rmse': rmse,
        'total_actual': date_data['actual'].sum(),
        'total_forecast': date_data['proposed_forecast'].sum()
    })
daily_metrics_df = pd.DataFrame(daily_metrics).sort_values('date')

fig, axes = plt.subplots(2, 1, figsize=(15, 8))

# 1. MAE and RMSE over time
axes[0].plot(daily_metrics_df['date'], daily_metrics_df['mae'], 
             marker='o', label='MAE', linewidth=2)
axes[0].plot(daily_metrics_df['date'], daily_metrics_df['rmse'], 
             marker='s', label='RMSE', linewidth=2)
axes[0].set_title('Forecast Accuracy Over Time', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Date')
axes[0].set_ylabel('Error Metric')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 2. Total demand: Actual vs Forecast
axes[1].plot(daily_metrics_df['date'], daily_metrics_df['total_actual'], 
             marker='o', label='Actual', linewidth=2)
axes[1].plot(daily_metrics_df['date'], daily_metrics_df['total_forecast'], 
             marker='s', label='Forecast', linewidth=2, linestyle='--')
axes[1].set_title('Daily Total Demand: Actual vs Forecast', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Date')
axes[1].set_ylabel('Total Units')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("  ✓ Time-based accuracy visualizations complete")

In [ ]:
# ===================================================================
# VISUALIZATION 3: ERROR ANALYSIS BY SEGMENT
# ===================================================================

print("\nCreating error analysis visualizations...")

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 1. MAE by segment
segment_mae = []
for segment in predictions_df['segment'].unique():
    seg_data = predictions_df[predictions_df['segment'] == segment]
    mae = mean_absolute_error(seg_data['actual'], seg_data['proposed_forecast'])
    segment_mae.append({'segment': segment, 'mae': mae})
segment_mae_df = pd.DataFrame(segment_mae).sort_values('mae')

axes[0, 0].barh(range(len(segment_mae_df)), segment_mae_df['mae'].values, alpha=0.7)
axes[0, 0].set_yticks(range(len(segment_mae_df)))
axes[0, 0].set_yticklabels(segment_mae_df['segment'])
axes[0, 0].set_title('MAE by Product Segment', fontsize=11, fontweight='bold')
axes[0, 0].set_xlabel('Mean Absolute Error')
axes[0, 0].grid(True, alpha=0.3, axis='x')

# 2. Error distribution by segment
segment_labels = predictions_df['segment'].unique()
segment_errors = [predictions_df[predictions_df['segment'] == seg]['proposed_forecast'] - 
                  predictions_df[predictions_df['segment'] == seg]['actual'] 
                  for seg in segment_labels]
axes[0, 1].boxplot(segment_errors, labels=segment_labels)
axes[0, 1].set_title('Error Distribution by Segment', fontsize=11, fontweight='bold')
axes[0, 1].set_xlabel('Segment')
axes[0, 1].set_ylabel('Prediction Error')
axes[0, 1].tick_params(axis='x', rotation=45)
axes[0, 1].axhline(y=0, color='r', linestyle='--', alpha=0.5)
axes[0, 1].grid(True, alpha=0.3, axis='y')

# 3. Method usage distribution
method_counts = predictions_df['method'].value_counts()
axes[1, 0].pie(method_counts.values, labels=method_counts.index, autopct='%1.1f%%',
               startangle=90)
axes[1, 0].set_title('Forecasting Method Distribution', fontsize=11, fontweight='bold')

# 4. Accuracy by method
method_accuracy = []
for method in predictions_df['method'].unique():
    method_data = predictions_df[predictions_df['method'] == method]
    mae = mean_absolute_error(method_data['actual'], method_data['proposed_forecast'])
    method_accuracy.append({'method': method, 'mae': mae, 'count': len(method_data)})
method_acc_df = pd.DataFrame(method_accuracy).sort_values('mae')

x = np.arange(len(method_acc_df))
width = 0.35
axes[1, 1].bar(x - width/2, method_acc_df['mae'].values, width, label='MAE', alpha=0.8)
axes[1, 1].bar(x + width/2, method_acc_df['count'].values/100, width, label='Count/100', alpha=0.8)
axes[1, 1].set_xticks(x)
axes[1, 1].set_xticklabels(method_acc_df['method'], rotation=45, ha='right')
axes[1, 1].set_title('MAE by Forecasting Method', fontsize=11, fontweight='bold')
axes[1, 1].set_ylabel('Value')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("  ✓ Error analysis visualizations complete")

## ADDITIONAL VISUALIZATIONS